In [11]:
# =============================================================================
# USB Charging Threat Detection — Behaviour-Based Security System
#  v6.0.2
#
# Research Project : "Behaviour-Based Detection of USB Charging Threats
#                     in Smart Devices"
# Threat Model     : Juice Jacking, BadUSB HID Spoofing, Passive Reconnaissance
# Architecture     : Data-First Synthetic Simulation → Calibrated ML Models
# Framework        : scikit-learn | numpy | pandas | SHAP | matplotlib
# Author           : Aryaveer Singh Parihar
# Reproducibility  : All stochastic operations seeded at RANDOM_STATE = 42
#

# Runtime  : ~8–15 minutes (includes adversarial testing + anomaly detection)
# =============================================================================

import warnings
import numpy as np
import pandas as pd
import pickle
import time
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, RandomizedSearchCV)
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, accuracy_score, precision_recall_curve,
                             average_precision_score)

# XAI & Visualization Libraries
import shap
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for Colab/server environments
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')


# ── Global Configuration ───────────────────────────────────────────────────
RANDOM_STATE = 42
N_SAMPLES    = 25_000
TARGET_NAMES = ['Class 0: Normal', 'Class 1: Suspicious', 'Class 2: Malicious']
CLASS_MAP    = {0: 'Normal', 1: 'Suspicious', 2: 'Malicious'}

# Seed the legacy numpy global RNG
np.random.seed(RANDOM_STATE)


# ============================================================================
#
#  ADVANCED DATA ENGINEERING & BEHAVIOURAL SIMULATION
# =============================================================================

print("=" * 72)
print("  PHASE 1 — Advanced Data Engineering & Behavioural Simulation")
print("=" * 72)

# ── Class Distribution ─────────────────────────────────────────────────────
n_normal     = int(N_SAMPLES * 0.55)               # 13,750
n_suspicious = int(N_SAMPLES * 0.28)               # 7,000
n_malicious  = N_SAMPLES - n_normal - n_suspicious  # 4,250

print(f"\n  Generating {N_SAMPLES:,} synthetic USB charging session records ...")
print(f"    Class 0 — Normal     : {n_normal:>6,}  ({n_normal     / N_SAMPLES:.1%})")
print(f"    Class 1 — Suspicious : {n_suspicious:>6,}  ({n_suspicious / N_SAMPLES:.1%})")
print(f"    Class 2 — Malicious  : {n_malicious:>6,}  ({n_malicious  / N_SAMPLES:.1%})")


def generate_normal_sessions(n: int) -> pd.DataFrame:
    """CLASS 0 — Standard Benign Wall Charger Sessions."""
    rng = np.random.default_rng(RANDOM_STATE)
    df = pd.DataFrame()

    df['session_duration_sec'] = rng.lognormal(mean=7.5, sigma=0.65, size=n)
    df['time_to_first_data_req_ms'] = 0.0
    df['voltage_fluctuation_mV'] = np.abs(rng.normal(loc=12.5, scale=3.2, size=n))
    df['current_draw_variance_mA'] = np.abs(rng.normal(loc=45.0, scale=14.0, size=n))
    df['power_drop_events_count'] = rng.poisson(lam=0.8, size=n)
    df['avg_energy_consumption_watts'] = np.clip(
        rng.normal(loc=18.0, scale=6.5, size=n), 4.5, 65.0
    )
    df['total_data_requests']       = rng.poisson(lam=0.40, size=n)
    df['unexpected_packet_count']   = rng.poisson(lam=0.15, size=n)
    df['device_info_requests']      = rng.poisson(lam=0.05, size=n)
    df['failed_handshake_attempts'] = rng.poisson(lam=0.25, size=n)
    df['usb_endpoint_registrations'] = np.where(
        rng.random(n) < 0.04, rng.choice([1, 2], size=n), 1
    ).astype(int)
    df['dynamic_class_swaps'] = rng.poisson(lam=0.05, size=n)
    df['packet_payload_entropy'] = np.clip(
        rng.normal(loc=0.6, scale=0.25, size=n), 0.0, 2.5
    )
    df['label'] = 0
    return df


def generate_suspicious_sessions(n: int) -> pd.DataFrame:
    """CLASS 1 — Passive Reconnaissance & Anomalous Power Signature Sessions."""
    rng = np.random.default_rng(RANDOM_STATE + 1)
    df = pd.DataFrame()

    df['session_duration_sec'] = rng.lognormal(mean=7.2, sigma=0.80, size=n)
    df['time_to_first_data_req_ms'] = np.clip(
        rng.normal(loc=350.0, scale=120.0, size=n), 0.0, 2000.0
    )
    df['voltage_fluctuation_mV'] = np.abs(rng.normal(loc=28.0, scale=7.5, size=n))
    df['current_draw_variance_mA'] = np.abs(rng.normal(loc=115.0, scale=35.0, size=n))
    df['power_drop_events_count'] = rng.poisson(lam=3.5, size=n)
    df['avg_energy_consumption_watts'] = np.clip(
        rng.normal(loc=14.0, scale=5.5, size=n), 3.0, 45.0
    )
    df['total_data_requests']       = np.clip(rng.poisson(lam=7.0, size=n), 1, 40)
    df['unexpected_packet_count']   = rng.poisson(lam=4.0, size=n)
    df['device_info_requests']      = rng.poisson(lam=2.0, size=n)
    df['failed_handshake_attempts'] = rng.poisson(lam=2.5, size=n)
    df['usb_endpoint_registrations'] = rng.choice(
        [1, 2, 3], size=n, p=[0.20, 0.65, 0.15]
    ).astype(int)
    df['dynamic_class_swaps'] = rng.poisson(lam=0.8, size=n)
    df['packet_payload_entropy'] = np.clip(
        rng.normal(loc=3.8, scale=1.0, size=n), 0.5, 7.0
    )

    # Stealth Phase: "Low-and-Slow" Masquerade (18%)
    stealth_mask = rng.random(n) < 0.18
    n_s = stealth_mask.sum()
    if n_s > 0:
        df.loc[stealth_mask, 'total_data_requests']       = rng.poisson(lam=0.40, size=n_s)
        df.loc[stealth_mask, 'voltage_fluctuation_mV']    = np.abs(rng.normal(13.0, 3.5, n_s))
        df.loc[stealth_mask, 'current_draw_variance_mA']  = np.abs(rng.normal(48.0, 15.0, n_s))
        df.loc[stealth_mask, 'packet_payload_entropy']    = np.clip(rng.normal(0.8, 0.3, n_s), 0.0, 2.5)
        df.loc[stealth_mask, 'unexpected_packet_count']   = rng.poisson(lam=0.20, size=n_s)
        df.loc[stealth_mask, 'device_info_requests']      = rng.poisson(lam=0.10, size=n_s)

    df['label'] = 1
    return df


def generate_malicious_sessions(n: int) -> pd.DataFrame:
    """CLASS 2 — Active BadUSB HID Spoofing & Aggressive Juice Jacking Sessions."""
    rng = np.random.default_rng(RANDOM_STATE + 2)
    df = pd.DataFrame()

    df['session_duration_sec'] = rng.lognormal(mean=6.8, sigma=1.0, size=n)
    df['time_to_first_data_req_ms'] = np.clip(
        rng.normal(loc=45.0, scale=15.0, size=n), 5.0, 150.0
    )
    df['voltage_fluctuation_mV'] = np.abs(rng.normal(loc=62.5, scale=9.5, size=n))
    df['current_draw_variance_mA'] = np.abs(rng.normal(loc=255.0, scale=68.0, size=n))
    df['power_drop_events_count'] = rng.poisson(lam=7.5, size=n)
    df['avg_energy_consumption_watts'] = np.clip(
        rng.normal(loc=9.5, scale=4.0, size=n), 2.0, 35.0
    )
    df['total_data_requests']       = np.clip(rng.poisson(lam=45.0, size=n), 15, 200)
    df['unexpected_packet_count']   = np.clip(rng.poisson(lam=22.0, size=n), 5,  150)
    df['device_info_requests']      = np.clip(rng.poisson(lam=8.0,  size=n), 1,   50)
    df['failed_handshake_attempts'] = np.clip(rng.poisson(lam=6.5,  size=n), 0,   60)
    df['usb_endpoint_registrations'] = np.clip(
        rng.poisson(lam=3.8, size=n), 3, 8
    ).astype(int)
    df['dynamic_class_swaps'] = np.clip(rng.poisson(lam=3.5, size=n), 1, 15)
    df['packet_payload_entropy'] = np.clip(
        rng.normal(loc=6.8, scale=0.7, size=n), 4.0, 8.0
    )

    # Stealth Phase: Pre-Attack Reconnaissance Window (12%)
    stealth_mask = rng.random(n) < 0.12
    n_s = stealth_mask.sum()
    if n_s > 0:
        df.loc[stealth_mask, 'voltage_fluctuation_mV']     = np.abs(rng.normal(13.0,  3.5,  n_s))
        df.loc[stealth_mask, 'current_draw_variance_mA']   = np.abs(rng.normal(46.0,  14.0, n_s))
        df.loc[stealth_mask, 'total_data_requests']        = rng.poisson(lam=0.40, size=n_s)
        df.loc[stealth_mask, 'time_to_first_data_req_ms']  = 0.0
        df.loc[stealth_mask, 'unexpected_packet_count']    = rng.poisson(lam=0.15, size=n_s)
        df.loc[stealth_mask, 'packet_payload_entropy']     = np.clip(
            rng.normal(0.8, 0.3, n_s), 0.0, 2.5
        )
        df.loc[stealth_mask, 'usb_endpoint_registrations'] = rng.choice([1, 2], size=n_s)
        df.loc[stealth_mask, 'dynamic_class_swaps']        = rng.poisson(lam=0.10, size=n_s)
        df.loc[stealth_mask, 'device_info_requests']       = rng.poisson(lam=0.05, size=n_s)
        df.loc[stealth_mask, 'failed_handshake_attempts']  = rng.poisson(lam=0.25, size=n_s)

    df['label'] = 2
    return df


# ── Generate Per-Class DataFrames ──────────────────────────────────────────
print("\n  Calling class-specific simulation engines ...")
df_normal     = generate_normal_sessions(n_normal)
df_suspicious = generate_suspicious_sessions(n_suspicious)
df_malicious  = generate_malicious_sessions(n_malicious)
print(f"    Normal sessions generated     : {len(df_normal):,}")
print(f"    Suspicious sessions generated : {len(df_suspicious):,}")
print(f"    Malicious sessions generated  : {len(df_malicious):,}")

# ── Concatenate and Shuffle ────────────────────────────────────────────────
df = pd.concat([df_normal, df_suspicious, df_malicious], ignore_index=True)
shuffle_idx = np.random.RandomState(RANDOM_STATE).permutation(N_SAMPLES)
df = df.iloc[shuffle_idx].reset_index(drop=True)

# ── Physical Consistency Enforcement ──────────────────────────────────────
df.loc[df['total_data_requests'] == 0, 'time_to_first_data_req_ms'] = 0.0

count_cols = [
    'power_drop_events_count', 'total_data_requests',
    'unexpected_packet_count', 'device_info_requests',
    'failed_handshake_attempts', 'usb_endpoint_registrations',
    'dynamic_class_swaps'
]
for col in count_cols:
    df[col] = np.maximum(df[col].values, 0).astype(int)

df['usb_endpoint_registrations'] = df['usb_endpoint_registrations'].clip(lower=1)
df['label'] = df['label'].astype(int)

# ── Gaussian Sensor Noise Layer ───────────────────────────────────────────
rng_noise = np.random.RandomState(RANDOM_STATE + 99)
continuous_cols = [
    'session_duration_sec', 'time_to_first_data_req_ms',
    'voltage_fluctuation_mV', 'current_draw_variance_mA',
    'avg_energy_consumption_watts', 'packet_payload_entropy'
]
for col in continuous_cols:
    q75   = df[col].quantile(0.75)
    q25   = df[col].quantile(0.25)
    iqr   = max(q75 - q25, 1e-6)
    noise = rng_noise.normal(0.0, iqr * 0.015, size=len(df))
    df[col] = np.maximum(df[col] + noise, 0.0)

df.loc[df['total_data_requests'] == 0, 'time_to_first_data_req_ms'] = 0.0

# ── Synthetic NaN Injection ────────────────────────────────────────────────
non_zero_idx = df[df['time_to_first_data_req_ms'] > 0].index
nan_count    = int(len(non_zero_idx) * 0.03)
nan_idx      = np.random.RandomState(RANDOM_STATE + 7).choice(
    non_zero_idx, size=nan_count, replace=False
)
df.loc[nan_idx, 'time_to_first_data_req_ms'] = np.nan

# ── Phase 1 Summary ────────────────────────────────────────────────────────
print(f"\n  [✓] Dataset generation complete.")
print(f"      Rows      : {len(df):,}   Features: {len(df.columns) - 1}   Target: 'label'")

print(f"\n  [✓] Final class distribution (post-shuffle):")
vc = df['label'].value_counts().sort_index()
for cls_id, cnt in vc.items():
    print(f"        Class {cls_id} — {CLASS_MAP[cls_id]:<12}: {cnt:>6,}  ({cnt/len(df):.2%})")


# =============================================================================
#
#  PREPROCESSING & GENERALISATION PIPELINE
# =============================================================================

print("\n" + "=" * 72)
print("  PHASE 2 — Preprocessing & Generalisation Pipeline")
print("=" * 72)

FEATURE_COLS = [c for c in df.columns if c != 'label']
X = df[FEATURE_COLS].copy()
y = df['label'].values.astype(int)

# ── Missing Value Imputation ───────────────────────────────────────────────
nan_before = X['time_to_first_data_req_ms'].isna().sum()
X['time_to_first_data_req_ms'] = X['time_to_first_data_req_ms'].fillna(-1.0)
nan_after = X.isna().sum().sum()

print(f"\n  [✓] NaN Imputation Complete:")
print(f"      Feature 'time_to_first_data_req_ms': {nan_before:,} NaN → sentinel -1.0")
print(f"      Remaining NaN values: {nan_after}")
assert nan_after == 0

# ── Stratified Train / Test Split (80 / 20) ───────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print(f"\n  [✓] Stratified 80/20 Train/Test Split (random_state={RANDOM_STATE}):")
print(f"      Training rows : {len(X_train):,}   |   Test rows : {len(X_test):,}")

# ── Cross-Validation Configuration ────────────────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f"\n  [✓] Stratified 5-Fold CV configured.")


# =============================================================================
#
#  HYPERPARAMETER OPTIMIZATION & CALIBRATED MODEL TRAINING
# =============================================================================

print("\n" + "=" * 72)
print("  PHASE 3 — Hyperparameter Optimization & Calibrated Training")
print("=" * 72)

# ── RandomizedSearchCV for Random Forest ──────────────────────────────────
print("\n  ── HYPERPARAMETER OPTIMIZATION: Random Forest ───────────────────")
print("     Using RandomizedSearchCV (n_iter=20, cv=3, scoring='f1_weighted')")

param_distributions = {
    'clf__n_estimators':      [100, 300, 500],
    'clf__max_depth':         [6, 8, 12, None],
    'clf__min_samples_leaf':  [5, 12, 20],
    'clf__max_features':      ['sqrt', 'log2'],
    'clf__class_weight':      ['balanced']
}

rf_base_pipeline = Pipeline([
    ('scaler', RobustScaler()),
    ('clf', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

skf_search = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

random_search = RandomizedSearchCV(
    estimator=rf_base_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    cv=skf_search,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=0
)

print("     Executing RandomizedSearchCV ...")
random_search.fit(X_train, y_train)

print(f"\n     [✓] OPTIMIZATION COMPLETE")
print(f"     Best CV F1-Weighted Score: {random_search.best_score_:.4f}")
print(f"     Optimal Hyperparameters:")
for param, value in random_search.best_params_.items():
    print(f"       {param:<30} : {value}")

rf_pipeline_uncalibrated = random_search.best_estimator_

# Store uncalibrated CV scores for reporting
rf_cv_scores_uncalibrated = cross_val_score(
    rf_pipeline_uncalibrated, X_train, y_train,
    cv=skf, scoring='f1_weighted', n_jobs=-1
)
print(f"\n     Uncalibrated 5-Fold CV Mean: {rf_cv_scores_uncalibrated.mean():.4f} "
      f"± {rf_cv_scores_uncalibrated.std():.4f}")


# ── Probability Calibration ───────────────────────────────────────────────
print("\n  ── PROBABILITY CALIBRATION ──────────────────────────────────────")
print("     Wrapping optimized Random Forest in CalibratedClassifierCV")
print("     Method: Isotonic Regression (non-parametric, monotonic)")
print("     Strategy: cv=5 (cross-validated calibration)")

calibrated_rf_pipeline = CalibratedClassifierCV(
    estimator=rf_pipeline_uncalibrated,
    method='isotonic',
    cv=5,
    n_jobs=-1
)

print("     Fitting calibration on training set ...")
calibrated_rf_pipeline.fit(X_train, y_train)
print("     [✓] CALIBRATION COMPLETE")

rf_cv_scores = rf_cv_scores_uncalibrated


# ── Model 2: Support Vector Machine (RBF Kernel) ──────────────────────────
print("\n  ── Model 2: Support Vector Machine — RBF Kernel ─────────────────")
svm_pipeline = Pipeline([
    ('scaler', RobustScaler()),
    ('clf', SVC(
        kernel='rbf', C=2.0, gamma='scale',
        class_weight='balanced', random_state=RANDOM_STATE,
        probability=True
    ))
])

svm_cv_scores = cross_val_score(
    svm_pipeline, X_train, y_train,
    cv=skf, scoring='f1_weighted', n_jobs=-1
)
print(f"     CV Mean ± Std: {svm_cv_scores.mean():.4f} ± {svm_cv_scores.std():.4f}")
svm_pipeline.fit(X_train, y_train)
print("     [✓] TRAINED")


# ── Model 3: Decision Tree Classifier ─────────────────────────────────────
print("\n  ── Model 3: Decision Tree — Interpretable Baseline ──────────────")
dt_pipeline = Pipeline([
    ('scaler', RobustScaler()),
    ('clf', DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=15, criterion='gini',
        class_weight='balanced', random_state=RANDOM_STATE
    ))
])

dt_cv_scores = cross_val_score(
    dt_pipeline, X_train, y_train,
    cv=skf, scoring='f1_weighted', n_jobs=-1
)
print(f"     CV Mean ± Std: {dt_cv_scores.mean():.4f} ± {dt_cv_scores.std():.4f}")
dt_pipeline.fit(X_train, y_train)
print("     [✓] TRAINED")


# =============================================================================

#  EVALUATION & OPERATIONAL THRESHOLD TUNING
# =============================================================================

print("\n" + "=" * 72)
print("  PHASE 4 — Evaluation & Operational Threshold Tuning")
print("=" * 72)


def evaluate_model(
    model_name: str,
    pipeline,
    cv_scores: np.ndarray,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
    feature_names: list = None,
    show_feature_importance: bool = False,
    is_calibrated: bool = False
) -> None:
    """Comprehensive evaluation with security-oriented metrics."""
    divider = "─" * 72
    print(f"\n  {divider}")
    print(f"  MODEL  :  {model_name}")
    print(f"  {divider}")

    cv_mean = cv_scores.mean()
    cv_std  = cv_scores.std()

    if cv_std < 0.008:
        stability_label = "HIGHLY STABLE   ✓✓"
    elif cv_std < 0.015:
        stability_label = "STABLE          ✓ "
    elif cv_std < 0.030:
        stability_label = "ACCEPTABLE      ~ "
    else:
        stability_label = "HIGH VARIANCE   ✗ "

    print(f"\n  [CV] Stratified 5-Fold Cross-Validation (F1-Weighted):")
    print(f"       Mean Score   :  {cv_mean:.4f}")
    print(f"       Std Dev      :  {cv_std:.4f}")
    print(f"       Stability    :  {stability_label}")

    if is_calibrated:
        print(f"       Note: CV scores from uncalibrated base model (calibration via cv=5)")

    y_pred = pipeline.predict(X_test)

    report_str = classification_report(
        y_test, y_pred, target_names=TARGET_NAMES, digits=4
    )

    print(f"\n  [REPORT] Classification Report — Hold-Out Test Set:")
    for line in report_str.strip().split('\n'):
        print(f"    {line}")

    cm = confusion_matrix(y_test, y_pred)

    print(f"\n  [MATRIX] Confusion Matrix")
    print(f"           Rows = Actual  |  Columns = Predicted")
    print()
    header = (f"  {'Actual \\ Predicted':<22}  {'→ Normal':>10}  "
              f"{'→ Suspicious':>14}  {'→ Malicious':>12}")
    print(header)
    print(f"  {'─' * 22}  {'─' * 10}  {'─' * 14}  {'─' * 12}")

    act_labels = ['↓ Normal      ', '↓ Suspicious  ', '↓ Malicious   ']
    for i, row_lbl in enumerate(act_labels):
        row = cm[i]
        cells = []
        for j, val in enumerate(row):
            cell_str = f"[{val:>5}]" if i == j else f" {val:>6} "
            cells.append(cell_str)
        print(f"  {row_lbl:<22}  {cells[0]:>10}  {cells[1]:>14}  {cells[2]:>12}")

    total_malicious  = cm[2].sum()
    tp_malicious     = cm[2][2]
    fn_malicious     = total_malicious - tp_malicious

    print(f"\n  [SECURITY] Threat Containment:")
    print(f"    Malicious Attack Detection Rate: {tp_malicious/total_malicious:.4f}")
    print(f"    Missed Malicious Attacks (FN)  : {fn_malicious} sessions "
          f"({fn_malicious/total_malicious:.2%})")

    if show_feature_importance and feature_names is not None:
        if is_calibrated and hasattr(pipeline, 'calibrated_classifiers_'):
            base_est = pipeline.calibrated_classifiers_[0].estimator
            clf_step = base_est.named_steps.get('clf')
        elif hasattr(pipeline, 'named_steps'):
            clf_step = pipeline.named_steps.get('clf')
        else:
            clf_step = None

        if clf_step is not None and hasattr(clf_step, 'feature_importances_'):
            importances = clf_step.feature_importances_
            sorted_idx  = np.argsort(importances)[::-1]

            print(f"\n  [FEATURES] Feature Importance Ranking (Gini Impurity)")
            print(f"    {'Rank':<5}  {'Feature Name':<36}  {'Importance':>11}  {'Cumulative':>11}")
            print(f"    {'─'*5}  {'─'*36}  {'─'*11}  {'─'*11}")

            cumulative = 0.0
            for rank, idx in enumerate(sorted_idx, start=1):
                feat = feature_names[idx]
                imp  = importances[idx]
                cumulative += imp
                print(f"    {rank:<5}  {feat:<36}  {imp:>11.6f}  {cumulative:>10.2%}")


# ── Evaluate All Models ────────────────────────────────────────────────────
print(f"\n  Evaluating all models on {len(y_test):,} test samples ...")

evaluate_model(
    model_name="Random Forest (Calibrated, Optimized)",
    pipeline=calibrated_rf_pipeline,
    cv_scores=rf_cv_scores,
    X_test=X_test,
    y_test=y_test,
    feature_names=FEATURE_COLS,
    show_feature_importance=True,
    is_calibrated=True
)

evaluate_model(
    model_name="SVM — RBF Kernel",
    pipeline=svm_pipeline,
    cv_scores=svm_cv_scores,
    X_test=X_test,
    y_test=y_test,
    show_feature_importance=False
)

evaluate_model(
    model_name="Decision Tree (Interpretable Baseline)",
    pipeline=dt_pipeline,
    cv_scores=dt_cv_scores,
    X_test=X_test,
    y_test=y_test,
    show_feature_importance=False
)


# ── OPERATIONAL THRESHOLD TUNING ──────────────────────────────────────────
print("\n" + "=" * 72)
print("  OPERATIONAL THRESHOLD TUNING (Malicious Class)")
print("=" * 72)

y_proba_calibrated = calibrated_rf_pipeline.predict_proba(X_test)
y_proba_malicious = y_proba_calibrated[:, 2]
y_test_binary = (y_test == 2).astype(int)

precision, recall, thresholds = precision_recall_curve(
    y_test_binary, y_proba_malicious
)

avg_precision = average_precision_score(y_test_binary, y_proba_malicious)

print(f"\n  [ANALYSIS] Precision-Recall Analysis for Malicious Class (Class 2)")
print(f"    Average Precision Score: {avg_precision:.4f}")

f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 0.5
optimal_precision = precision[optimal_idx]
optimal_recall = recall[optimal_idx]
optimal_f1 = f1_scores[optimal_idx]

print(f"\n  [OPTIMAL THRESHOLD] Security-Oriented Operating Point:")
print(f"    Threshold (P(Malicious))   : {optimal_threshold:.4f}")
print(f"    Precision @ Threshold      : {optimal_precision:.4f}")
print(f"    Recall @ Threshold         : {optimal_recall:.4f}")
print(f"    F1-Score @ Threshold       : {optimal_f1:.4f}")

recall_95_idx = np.where(recall >= 0.95)[0]
if len(recall_95_idx) > 0:
    threshold_95_recall = thresholds[recall_95_idx[0]] if recall_95_idx[0] < len(thresholds) else 0.5
    precision_95_recall = precision[recall_95_idx[0]]
    print(f"\n  [ALTERNATIVE] High-Security Operating Point (95% Recall):")
    print(f"    Threshold (P(Malicious))   : {threshold_95_recall:.4f}")
    print(f"    Precision @ 95% Recall     : {precision_95_recall:.4f}")
    print(f"    Recall                     : ≥0.95")

print(f"\n  Generating Precision-Recall Curve plot ...")
plt.figure(figsize=(10, 7), dpi=150)
plt.plot(recall, precision, linewidth=2, label=f'PR Curve (AP={avg_precision:.3f})')
plt.scatter(optimal_recall, optimal_precision, color='red', s=100, zorder=5,
            label=f'Optimal Threshold={optimal_threshold:.3f}')
plt.xlabel('Recall (True Positive Rate)', fontsize=12)
plt.ylabel('Precision (Positive Predictive Value)', fontsize=12)
plt.title('Precision-Recall Curve — Malicious Class (Class 2)\n'
          'Operational Threshold Optimization for Security Applications',
          fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.savefig('precision_recall_curve.png', dpi=150, bbox_inches='tight')
plt.close()
print("  [✓] Saved: precision_recall_curve.png")


# ── SHAP ANALYSIS ──────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print("  EXPLAINABLE AI (XAI) — SHAP Analysis (Robust Modern API)")
print("=" * 72)

print("\n  Initializing SHAP TreeExplainer for calibrated Random Forest ...")
base_estimator = calibrated_rf_pipeline.calibrated_classifiers_[0].estimator
rf_clf = base_estimator.named_steps['clf']
scaler = base_estimator.named_steps['scaler']
X_test_scaled = scaler.transform(X_test)

print("  Creating TreeExplainer ...")
explainer = shap.TreeExplainer(rf_clf)

print("  Computing SHAP values for test set (500 samples) ...")
X_test_subset = X_test_scaled[:500]
shap_values = explainer.shap_values(X_test_subset)

print("  [✓] SHAP values computed successfully")

print("\n  Extracting Malicious class (Class 2) SHAP values ...")

if isinstance(shap_values, np.ndarray) and len(shap_values.shape) == 3:
    print(f"      Detected: Modern SHAP 3D array (shape: {shap_values.shape})")
    shap_values_malicious = shap_values[:, :, 2]
    print(f"      Extracted Class 2 shape: {shap_values_malicious.shape}")

elif isinstance(shap_values, list):
    print(f"      Detected: Legacy SHAP list format ({len(shap_values)} classes)")
    shap_values_malicious = shap_values[2]
    print(f"      Extracted Class 2 shape: {shap_values_malicious.shape}")

else:
    raise TypeError(f"Unexpected SHAP values type: {type(shap_values)}")

print("  [✓] Class 2 (Malicious) SHAP values extracted successfully")

print("\n  Generating SHAP Summary Plot (Beeswarm) ...")
plt.figure(figsize=(12, 8), dpi=150)
shap.summary_plot(
    shap_values_malicious,
    X_test_subset,
    feature_names=FEATURE_COLS,
    show=False,
    max_display=13
)
plt.title("SHAP Summary Plot — Malicious Class (Class 2)\n"
          "Feature Impact on Threat Classification Predictions",
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.close()
print("  [✓] Saved: shap_summary.png")

print("\n  Generating SHAP Feature Importance Bar Plot ...")
plt.figure(figsize=(10, 8), dpi=150)
shap.summary_plot(
    shap_values_malicious,
    X_test_subset,
    feature_names=FEATURE_COLS,
    plot_type="bar",
    show=False,
    max_display=13
)
plt.title("SHAP Feature Importance — Malicious Class (Class 2)\n"
          "Mean Absolute SHAP Value per Feature",
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print("  [✓] Saved: shap_importance.png")


# ============================================================================
#  EDGE DEVICE DEPLOYMENT BENCHMARKING
# =============================================================================

print("\n" + "=" * 72)
print("  PHASE 5 — Edge Device Deployment Benchmarking")
print("=" * 72)

print("\n  ── INFERENCE LATENCY PROFILING ──────────────────────────────────")
print("     Measuring prediction latency on Calibrated Random Forest ...")

n_iterations = 10
latencies = []

for i in range(n_iterations):
    start_time = time.perf_counter()
    _ = calibrated_rf_pipeline.predict(X_test)
    end_time = time.perf_counter()
    latencies.append(end_time - start_time)

latencies = np.array(latencies)
mean_total_time = latencies.mean()
std_total_time = latencies.std()

mean_latency_per_sample_sec = mean_total_time / len(X_test)
mean_latency_per_sample_us  = mean_latency_per_sample_sec * 1e6

print(f"\n     Benchmark Configuration:")
print(f"       Test Set Size    : {len(X_test):,} samples")
print(f"       Iterations       : {n_iterations}")
print(f"       Model            : Calibrated Random Forest")
print(f"\n     Results:")
print(f"       Total Inference Time (mean)  : {mean_total_time*1000:.2f} ms ± {std_total_time*1000:.2f} ms")
print(f"       Per-Sample Latency (mean)    : {mean_latency_per_sample_us:.2f} μs/sample")
print(f"       Throughput                    : {len(X_test)/mean_total_time:.0f} samples/second")

print("\n  ── MODEL MEMORY FOOTPRINT ANALYSIS ──────────────────────────────")
print("     Serializing Calibrated Random Forest ...")

serialized_model = pickle.dumps(calibrated_rf_pipeline)
model_size_bytes = len(serialized_model)
model_size_kb = model_size_bytes / 1024
model_size_mb = model_size_kb / 1024

print(f"\n     Memory Footprint:")
print(f"       Raw Bytes   : {model_size_bytes:,} bytes")
print(f"       Kilobytes   : {model_size_kb:.2f} KB")
print(f"       Megabytes   : {model_size_mb:.3f} MB")

print("\n  ── EDGE DEPLOYMENT FEASIBILITY REPORT ───────────────────────────")
print(f"""
  ┌─ Resource Profile ─────────────────────────────────────────────────┐
  │                                                                     │
  │  INFERENCE LATENCY:                                                 │
  │    Per-sample latency    : {mean_latency_per_sample_us:>8.2f} μs/sample                   │
  │    Real-time capability  : {'✓ PASS' if mean_latency_per_sample_us < 500 else '✗ FAIL'}  (target: <500 μs/sample)        │
  │                                                                     │
  │  MEMORY FOOTPRINT:                                                  │
  │    Model size            : {model_size_mb:>8.3f} MB                                │
  │    Embedded RAM budget   : {'✓ PASS' if model_size_mb < 15 else '✗ FAIL'}  (target: <15 MB w/ calibration)   │
  │                                                                     │
  │  DEPLOYMENT VERDICT:                                                │
  │    Smartphone (2GB+ RAM) : ✓ EXCELLENT — Production Ready           │
  │    IoT Device (<512 MB)  : {'✓ SUITABLE (with quantization)' if model_size_mb*0.4 < 6 else '~ MARGINAL'}      │
  │                                                                     │
  └─────────────────────────────────────────────────────────────────────┘
""")

print("\n" + "=" * 72)
print("  COMPARATIVE MODEL PERFORMANCE SUMMARY")
print("=" * 72)

models_summary = [
    ("Calibrated RF (Optimized)", calibrated_rf_pipeline, rf_cv_scores),
    ("SVM-RBF (C=2.0)          ", svm_pipeline, svm_cv_scores),
    ("Decision Tree (depth≤6)  ", dt_pipeline, dt_cv_scores),
]

print(f"\n  {'Model':<35}  {'CV F1-W':>9}  {'CV Std':>8}  "
      f"{'Test F1-W':>11}  {'Test Acc':>10}")
print(f"  {'─'*35}  {'─'*9}  {'─'*8}  {'─'*11}  {'─'*10}")

for name, pipe, cv_sc in models_summary:
    y_p = pipe.predict(X_test)
    test_f1  = f1_score(y_test, y_p, average='weighted')
    test_acc = accuracy_score(y_test, y_p)
    print(f"  {name:<35}  {cv_sc.mean():>9.4f}  {cv_sc.std():>8.4f}  "
          f"{test_f1:>11.4f}  {test_acc:>10.4f}")

print()


# =============================================================================
#
#
#  ADVERSARIAL EVASION SIMULATION (FIXED v6.0.2)
# =============================================================================

print("\n" + "=" * 72)
print("  PHASE 6 — ADVERSARIAL EVASION SIMULATION")
print("=" * 72)

print("\n  Simulating 'Smart Hacker' Attack Throttling Strategy ...")
print("  Threat Model: Advanced attacker throttles malicious signatures")
print("               to blend 30% toward Normal class statistical profile")

# ── Isolate Malicious Test Samples ────────────────────────────────────────
malicious_mask = (y_test == 2)
X_test_malicious = X_test[malicious_mask].copy()
y_test_malicious = y_test[malicious_mask].copy()

print(f"\n  [DATA] Isolated {len(X_test_malicious):,} Malicious test samples")

# ── Compute Normal Class Statistical Baselines ────────────────────────────
normal_train_mask = (y_train == 0)
X_train_normal = X_train[normal_train_mask]

# Calculate means for primary threat features
primary_threat_features = [
    'total_data_requests',
    'voltage_fluctuation_mV',
    'unexpected_packet_count'
]

normal_means = {}
for feat in primary_threat_features:
    normal_means[feat] = X_train_normal[feat].mean()

print(f"\n  [BASELINE] Normal Class Feature Means (from training data):")
for feat in primary_threat_features:
    print(f"    {feat:<30} : {normal_means[feat]:>8.2f}")

# ── Apply Adversarial Perturbation (30% Blend) ────────────────────────────
print(f"\n  [ATTACK] Applying 30% adversarial perturbation ...")

X_test_adversarial = X_test_malicious.copy()

for feat in primary_threat_features:
    # Blend: 70% original malicious value + 30% normal baseline
    X_test_adversarial[feat] = (
        0.70 * X_test_malicious[feat] +
        0.30 * normal_means[feat]
    )

print(f"  [✓] Adversarial dataset created: {len(X_test_adversarial):,} evasive samples")

# ── Evaluate Calibrated Model Against Evasive Attacks ─────────────────────
print(f"\n  Testing Calibrated Random Forest against evasive attacks ...")

y_pred_adversarial = calibrated_rf_pipeline.predict(X_test_adversarial)

# ── Classification Report with Proper Labels ──────────────────────────────
adversarial_report = classification_report(
    y_test_malicious,
    y_pred_adversarial,
    labels=[0, 1, 2],
    target_names=TARGET_NAMES,
    digits=4,
    zero_division=0
)

print(f"\n  ── ADVERSARIAL ROBUSTNESS REPORT ─────────────────────────────────")
print(f"\n  Classification Report — Evasive Malicious Attacks:")
for line in adversarial_report.strip().split('\n'):
    print(f"    {line}")

# ── FIXED v6.0.2: Corrected Variable Name ─────────────────────────────────
correctly_detected = (y_pred_adversarial == 2).sum()
evaded_as_normal = (y_pred_adversarial == 0).sum()
evaded_as_suspicious = (y_pred_adversarial == 1).sum()
total_evasive = len(y_test_malicious)  # FIXED: was y_test_adversarial

evasion_success_rate = (evaded_as_normal + evaded_as_suspicious) / total_evasive
detection_rate = correctly_detected / total_evasive

print(f"\n  ── EVASION SUMMARY ───────────────────────────────────────────────")
print(f"\n  Total Evasive Attacks                : {total_evasive:>4}")
print(f"  ─────────────────────────────────────────")
print(f"  Successfully Caught (Class 2)        : {correctly_detected:>4}  ({detection_rate:>6.2%})")
print(f"  Evaded as Suspicious (Class 1)       : {evaded_as_suspicious:>4}  ({evaded_as_suspicious/total_evasive:>6.2%})")
print(f"  Evaded as Normal (Class 0)           : {evaded_as_normal:>4}  ({evaded_as_normal/total_evasive:>6.2%})")
print(f"  ─────────────────────────────────────────")
print(f"  Total Evasion Success Rate           : {evasion_success_rate:>6.2%}")

print(f"\n  [ROBUSTNESS VERDICT]")
if detection_rate >= 0.75:
    verdict = "✓ ROBUST   — Model maintains 75%+ detection under evasion"
elif detection_rate >= 0.50:
    verdict = "~ MODERATE — Model shows 50-75% resilience to throttling"
else:
    verdict = "✗ WEAK     — Model highly vulnerable to adaptive attacks"

print(f"    {verdict}")


# =============================================================================
#
#  ZERO-DAY ANOMALY DETECTION
# =============================================================================

print("\n" + "=" * 72)
print("  PHASE 7 — ZERO-DAY ANOMALY DETECTION")
print("=" * 72)

print("\n  Training Unsupervised Isolation Forest on Normal Class Only ...")
print("  Goal: Detect unknown attack vectors without supervised labels")

# ── Train Isolation Forest Exclusively on Normal Training Data ────────────
X_train_normal_only = X_train[y_train == 0].copy()

print(f"\n  [DATA] Training samples (Normal only): {len(X_train_normal_only):,}")

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.10,
    max_samples='auto',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print(f"  [CONFIG] IsolationForest(n_estimators=200, contamination=0.10)")
print(f"  Training on Normal class baseline ...")

iso_forest.fit(X_train_normal_only)
print(f"  [✓] ZERO-DAY DETECTOR TRAINED")

# ── Create Mixed Hold-Out Evaluation Set ──────────────────────────────────
normal_test_mask = (y_test == 0)
malicious_test_mask = (y_test == 2)

X_test_normal = X_test[normal_test_mask]
X_test_malicious_zeroday = X_test[malicious_test_mask]

X_test_mixed = pd.concat([X_test_normal, X_test_malicious_zeroday], ignore_index=True)
y_test_mixed_binary = np.concatenate([
    np.zeros(len(X_test_normal)),
    np.ones(len(X_test_malicious_zeroday))
])

print(f"\n  [EVAL SET] Mixed hold-out test set:")
print(f"    Normal samples    : {len(X_test_normal):,}")
print(f"    Malicious samples : {len(X_test_malicious_zeroday):,}")
print(f"    Total             : {len(X_test_mixed):,}")

# ── Test Isolation Forest Anomaly Detection ───────────────────────────────
print(f"\n  Testing Isolation Forest on mixed Normal/Malicious data ...")

anomaly_predictions = iso_forest.predict(X_test_mixed)
anomaly_binary = (anomaly_predictions == -1).astype(int)

total_normal = len(X_test_normal)
total_malicious = len(X_test_malicious_zeroday)

malicious_indices = np.arange(total_normal, len(X_test_mixed))
tp_anomaly = anomaly_binary[malicious_indices].sum()

normal_indices = np.arange(0, total_normal)
fp_anomaly = anomaly_binary[normal_indices].sum()

tn_anomaly = total_normal - fp_anomaly
fn_anomaly = total_malicious - tp_anomaly

detection_rate_zeroday = tp_anomaly / total_malicious if total_malicious > 0 else 0
false_positive_rate = fp_anomaly / total_normal if total_normal > 0 else 0
precision_zeroday = tp_anomaly / (tp_anomaly + fp_anomaly) if (tp_anomaly + fp_anomaly) > 0 else 0
f1_zeroday = 2 * (precision_zeroday * detection_rate_zeroday) / (precision_zeroday + detection_rate_zeroday + 1e-10)

print(f"\n  ── ZERO-DAY DETECTION PERFORMANCE ────────────────────────────────")
print(f"\n  Confusion Matrix (Anomaly Detection):")
print(f"                      Predicted Normal  Predicted Anomaly")
print(f"    Actual Normal         {tn_anomaly:>6}             {fp_anomaly:>6}")
print(f"    Actual Malicious      {fn_anomaly:>6}             {tp_anomaly:>6}")

print(f"\n  [METRICS]")
print(f"    Zero-Day Detection Rate (Recall)  : {detection_rate_zeroday:.4f} ({tp_anomaly}/{total_malicious})")
print(f"    False Positive Rate               : {false_positive_rate:.4f} ({fp_anomaly}/{total_normal})")
print(f"    Precision                         : {precision_zeroday:.4f}")
print(f"    F1-Score                          : {f1_zeroday:.4f}")

print(f"\n  [ZERO-DAY VERDICT]")
if detection_rate_zeroday >= 0.70:
    zd_verdict = "✓ EXCELLENT — Isolation Forest detects 70%+ unknown threats"
elif detection_rate_zeroday >= 0.50:
    zd_verdict = "~ MODERATE  — 50-70% detection of novel attack patterns"
else:
    zd_verdict = "✗ WEAK      — Unsupervised approach needs tuning"

print(f"    {zd_verdict}")

print(f"\n  [INSIGHT] Isolation Forest provides complementary zero-day protection")
print(f"            by detecting statistical anomalies without labeled attack data.")



print("\n" + "=" * 72)
print("  PIPELINE EXECUTION COMPLETE")
print("=" * 72)
print(f"""


  ┌─ Advanced Security Analysis Results ───────────────────────────────────┐
  │  Adversarial Robustness  : {detection_rate:.2%} detection under 30% evasion         │
  │  Zero-Day Detection Rate : {detection_rate_zeroday:.2%} (unsupervised Isolation Forest)    │
  │  Optimal Threshold       : {optimal_threshold:.4f} (Precision-Recall optimized)         │
  └─────────────────────────────────────────────────────────────────────────┘

  ┌─ Artifacts Generated ──────────────────────────────────────────────────┐
  │  📊 shap_summary.png           — SHAP Beeswarm plot                    │
  │  📊 shap_importance.png        — SHAP feature ranking                  │
  │  📊 precision_recall_curve.png — Threshold optimization                         │
  └─────────────────────────────────────────────────────────────────────────┘

""")


print("\n" + "=" * 72 + "\n")

  PHASE 1 — Advanced Data Engineering & Behavioural Simulation

  Generating 25,000 synthetic USB charging session records ...
    Class 0 — Normal     : 13,750  (55.0%)
    Class 1 — Suspicious :  7,000  (28.0%)
    Class 2 — Malicious  :  4,250  (17.0%)

  Calling class-specific simulation engines ...
    Normal sessions generated     : 13,750
    Suspicious sessions generated : 7,000
    Malicious sessions generated  : 4,250

  [✓] Dataset generation complete.
      Rows      : 25,000   Features: 13   Target: 'label'

  [✓] Final class distribution (post-shuffle):
        Class 0 — Normal      : 13,750  (55.00%)
        Class 1 — Suspicious  :  7,000  (28.00%)
        Class 2 — Malicious   :  4,250  (17.00%)

  PHASE 2 — Preprocessing & Generalisation Pipeline

  [✓] NaN Imputation Complete:
      Feature 'time_to_first_data_req_ms': 364 NaN → sentinel -1.0
      Remaining NaN values: 0

  [✓] Stratified 80/20 Train/Test Split (random_state=42):
      Training rows : 20,000   |   Te